# Retrieval-augmented Generation

FINAL GOAL: design a RAG system from scratch.

Before proceeding, start a `.py` file (or a `.ipynb` file) to work with these activities.

## Exercise 1

Suppose we are trying to search some information within the materials from this course. The course is reasonably well organized, and everything is within jupyter notebooks. Jupyter notebook files (.ipynb) are actually in JSON format.

Choose any `.ipynb` file and open it using the `json` library in Python. Investigate:

1. How can we find a cell within the notebook?
1. How can we known if the cell contains Python code or Markdown annotations?

In [1]:
import json

# Open and load the .ipynb file
with open("../classic_nlp/03-finding_content.ipynb", "r", encoding="utf-8") as file:
    notebook = json.load(file)

# Now you can access the notebook content
print(type(notebook))           # Should be a dict
print(notebook.keys())          # Top-level keys like 'cells', 'metadata', 'nbformat'

<class 'dict'>
dict_keys(['cells', 'metadata', 'nbformat', 'nbformat_minor'])


In [2]:
notebook['cells'][0]['source']

['# Content-based search\n',
 '\n',
 'The Internet has brought forward a marvelous source of information. But - simply knowing that we *have* information is just not enough to *use* this information. For example, we *know* that, somewhere on the Internet, there is a book on Natural Language Processing. But, how can we find this book?\n',
 '\n',
 'In this notebook, we are going to work with the following use case (which was also approached in [Amami et al., "An LDA-Based Approach to Scientific Paper Recommendation",Natural Language Processing and Information Systems, 2016 ](http://link.springer.com/10.1007/978-3-319-41754-7_17), based on ideas by [Griffiths and Steyvers, "Finding Scientific Topics", Proc. Natl. Acad. Sci. U.S.A., 2004](https://doi.org/10.1073/pnas.0307752101).\n',
 '\n',
 'Suppose a scientist is writing an article. Articles usually start with a session called "abstract", which summarizes the contents of the whole paper. We want our system to get the abstract we are work

In [3]:
# For example, printing the source code of each code cell
for cell in notebook.get("cells", []):
    if cell.get("cell_type") == "code":
        print("Code cell:")
        print("".join(cell.get("source", [])))
        print("-" * 40)

Code cell:
import pandas as pd 
import os
import kagglehub
from tqdm import tqdm
from pathlib import Path
    
path = kagglehub.dataset_download("tiagoft/arvix-data-filtered-for-cs-only-data")
path = Path(path)
df = pd.read_csv(path / 'arxiv-metadata-oai-snaptshot-cs-only.csv')
----------------------------------------
Code cell:
sample_title = "Enhancing Autonomous Agents with Multimodal Generative AI for Improved Human-AI Collaboration"
sample_abstract = """The integration of multimodal generative AI into autonomous agents presents a significant advancement in human-AI collaboration. 
This study explores the development of autonomous agents capable of processing and generating various data types,
including text-to-image and image-to-audio conversions. By leveraging multimodal generative AI, these agents can interpret and generate 
content across different modalities, enhancing their ability to interact with humans in more natural and intuitive ways.
We propose a novel framework that c

## Exercise 2

The simplest way to search for text is using keywords. Improve your code so that it:

1. Collects a keyword from the user, and
1. Indicates all notebooks/cells that contain that keyword.

Reflect: what is the best way to present these results to the user?

In [4]:
# Para fins práticos, vamos considerar cada célula como um "documento"
# Assim, podemos usar document indexing
import re
from collections import defaultdict
from nltk.stem import WordNetLemmatizer

iidx = defaultdict(set)

def adicionar_ao_indice(iidx, texto_novo, idx, lemmatize=False):
    texto = texto_novo.upper()
    palavras = re.findall(r'\b\w\w+\b', texto)
    if lemmatize:
        lematizer = WordNetLemmatizer()
        palavras = [lematizer.lemmatize(w) for w in palavras]

    for p in palavras:
        iidx[p].add(idx)
    return iidx

for i, cell in enumerate(notebook.get("cells", [])):
    content = "".join(cell.get("source", []))
    iidx = adicionar_ao_indice(iidx, content, i)

print(iidx)

defaultdict(<class 'set'>, {'CONTENT': {0, 2, 5}, 'BASED': {0, 15}, 'SEARCH': {0, 3}, 'THE': {0, 2, 3, 6, 10, 12, 14, 15}, 'INTERNET': {0}, 'HAS': {0}, 'BROUGHT': {0}, 'FORWARD': {0}, 'MARVELOUS': {0}, 'SOURCE': {0}, 'OF': {0, 2, 6, 10, 11, 14, 15}, 'INFORMATION': {0}, 'BUT': {0}, 'SIMPLY': {0, 3}, 'KNOWING': {0}, 'THAT': {0, 2, 6, 10, 14}, 'WE': {0, 10, 2, 3}, 'HAVE': {0, 10}, 'IS': {0, 3, 10, 12, 14}, 'JUST': {0}, 'NOT': {0, 10, 12}, 'ENOUGH': {0}, 'TO': {0, 2, 3, 6, 10, 12, 14, 15}, 'USE': {0, 3, 15}, 'THIS': {0, 2, 6, 12, 15}, 'FOR': {0, 1, 2, 4, 5, 6, 10, 11, 12, 14}, 'EXAMPLE': {0}, 'KNOW': {0}, 'SOMEWHERE': {0}, 'ON': {0}, 'THERE': {0, 10}, 'BOOK': {0}, 'NATURAL': {0, 2}, 'LANGUAGE': {0}, 'PROCESSING': {0, 2}, 'HOW': {0, 10, 12, 15}, 'CAN': {0, 2, 3, 12, 14, 15}, 'FIND': {0, 3, 6, 10, 14}, 'IN': {0, 2, 3, 4, 5, 6, 10, 11, 12, 14}, 'NOTEBOOK': {0}, 'ARE': {0, 10, 6, 14}, 'GOING': {0}, 'WORK': {0}, 'WITH': {0, 2, 6, 10, 15}, 'FOLLOWING': {0}, 'CASE': {0}, 'WHICH': {0, 12, 15}, 'WA

In [5]:
def encontrar_no_indice(iidx, palavra, lemmatize=False):
    if lemmatize:
        lematizer = WordNetLemmatizer()
        palavra = lematizer.lemmatize(palavra)

    idx = iidx[palavra.upper()]
    return idx

# keyword = input()
keyword = "keyword"
docs_index = encontrar_no_indice(iidx, keyword)
docs_content = []
for i in list(docs_index):
    content = "".join(notebook['cells'][i].get("source", []))
    docs_content.append(content)

print(docs_content)

["## Exercise 1: search by keyword\n\nSearching by keywords is somewhat simple because we can simply use an inverted index. In fact, online search engines usually implement inverted index.\n\nUse your inverted index to try to find other, relevant articles within our dataset using the keywords provided by the abstract's author.", '## Exercise 5\n\nCompare the recommendations provided by keyword searching, by TDIDF keyword searching, and by topic modelling. \n\n1. Which recommendation seems more useful?\n1. Could you combine the techniques above (at least 2 of them) to get a possibly better recommendation?\n1. Can you use an LLM to help with this task? How? Implement an LLM-based solution and compare it with the previous ones.\n']


## Exercise 3

There is an inherent fragility in the previous system: it requires the user to guess the keyword correctly. Trivia fact: in the early 2000s, the hability to guess keywords in Google had the same hype that using AI chatbots has nowadays.

We could prevent our user from trying to guess the exact keyword or phrase, afterall, we have an estimator for phrase similarity: BERT!

Improve your code so that it:

1. Collects a phrase from the user,
1. Calculates the phrase embedding $q$ using the CLS token from BERT
1. Traverses the course material calculating the embedding $x_i$ for each cell
1. Finds the $k$ (try with $k=1$, then generalize to any $k$) cells with minimal cosine distance ($d = \frac{ <q, x_1>}{||x|| ||c_i||}$) with relationship to the phrase.

Reflect: was this a better choice for retrieval? How can we measure this difference? (tip: research how information retrieval systems are evaluated!)

In [6]:
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained("bert-base-uncased")

def get_embeddings(text, model, tokenizer):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=512)
    outputs = model(**inputs)
    cls_embedding = outputs.last_hidden_state[0, 0, :]
    return cls_embedding

In [7]:
phrase = "search for information using document indexing"
q = get_embeddings(phrase, model, tokenizer)

In [9]:
documents_embeddings = {}
for i, cell in tqdm(enumerate(notebook.get("cells", []))):
    content = "".join(notebook['cells'][i].get("source", []))
    cell_embedding = get_embeddings(content, model, tokenizer)
    documents_embeddings[i] = cell_embedding
documents_embeddings.keys()

16it [00:04,  3.98it/s]


dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15])

In [ ]:
import numpy as np
import torch.nn.functional as f

def find_cosine_distace(q, doc_emb):
    cos_sim = f.cosine_similarity(q.unsqueeze(0), doc_emb.unsqueeze(0))  # Add batch dimension
    d = 1 - cos_sim

    return float(d[0])

def find_k_cells(dist_cells, k, docs):
    all_scores = dist_cells.copy()
    all_scores.sort()

    contexts = []
    for _, i in all_scores[:k]:
        content = "".join(notebook['cells'][i].get("source", []))
        contexts.append(content)

    return contexts


dist_cells = []
for i in documents_embeddings.keys():
    d = find_cosine_distace(q, documents_embeddings[i])
    dist_cells.append([d, i])

['# Content-based search\n\nThe Internet has brought forward a marvelous source of information. But - simply knowing that we *have* information is just not enough to *use* this information. For example, we *know* that, somewhere on the Internet, there is a book on Natural Language Processing. But, how can we find this book?\n\nIn this notebook, we are going to work with the following use case (which was also approached in [Amami et al., "An LDA-Based Approach to Scientific Paper Recommendation",Natural Language Processing and Information Systems, 2016 ](http://link.springer.com/10.1007/978-3-319-41754-7_17), based on ideas by [Griffiths and Steyvers, "Finding Scientific Topics", Proc. Natl. Acad. Sci. U.S.A., 2004](https://doi.org/10.1073/pnas.0307752101).\n\nSuppose a scientist is writing an article. Articles usually start with a session called "abstract", which summarizes the contents of the whole paper. We want our system to get the abstract we are working with, and then find possib

## Exercise 4

Now let's leave our retrival system waiting for a while.

Make a small program that:

1. Collects a question from the user
1. Uses an API to redirect this question to an LLM, and immediately returns the answer.
1. Add prompt information so that your answers can only regard NLP-related subjects (these are called "safeguards")

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY')

question = "How can I search for information using document indexing?"

prompt = f"""
Answer the following question. Use the context provided (between <context></context> brackets)\
to answer the question.

question: {question}

<context>
context
</context>
"""

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash", # max 1500 / dia
    temperature=0.7,
)

response = llm.invoke(prompt)
response

AIMessage(content='The provided context is empty.  Therefore, I cannot answer how to search for information using document indexing.  To answer your question, I need information about a specific document indexing system or method.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash', 'safety_ratings': []}, id='run--a8dec1f4-567e-4a33-aaf6-e7592ae90ff0-0', usage_metadata={'input_tokens': 48, 'output_tokens': 40, 'total_tokens': 88, 'input_token_details': {'cache_read': 0}})

## Exercise 5

Now, let's joint everything.

We are able to find specific information from our courseware. Also, we are able to use LLMs. Use both abilities to:

1. Collect a question from the user
1. Retrieve the $K$ most relevant cells from the course material
1. Use the content of these cells as part of a prompt. The prompt includes both the question and the content from the relevant cells.
1. Phrase your prompt so that the LLM can only return information that is contained in the course material.

Reflect: how does this compare to the system in Exercise 4? How can we measure the differences?

In [26]:
question = "How can I search for information using document indexing?"
query = get_embeddings(question, model, tokenizer)

dist_docs = []
for i in documents_embeddings.keys():
    d = find_cosine_distace(query, documents_embeddings[i])
    dist_docs.append([d, i])

context_list = find_k_cells(dist_docs, 3, notebook)
context = "\n\n".join(context_list)

prompt = f"""
Answer the following question. Use the context provided (between <context></context> brackets)\
to answer the question.

question: {question}

<context>
{context}
</context>
"""

response = llm.invoke(prompt)
response.content

"The provided text describes searching using an inverted index, a method employed by online search engines.  To find relevant articles using this method, you would use the keywords from the abstract's author as your search terms.  The inverted index then allows for efficient retrieval of articles containing those keywords."

## Exercise 6

If you have reached this far, let's start optimizing our systems.

To do so:

1. Identify which step of your processing pipeline takes the longer
1. Study if there are techniques or data structures that can make this specific step faster
1. If possible, implement the optimization and test the results.
1. Iterate until you cannot optimize anymore.